# DaTSCAN — Fase 5: diagnóstico y calibración por protocolo

Este notebook audita las predicciones *out-of-fold* (OOF) de la CNN 3D local de dos canales. Sus objetivos son:

1. comprobar la integridad de las 1,362 predicciones;
2. medir discriminación y calibración global y por protocolo;
3. evaluar calibración global **cruzada**, sin ajustar y evaluar con los mismos pacientes;
4. estudiar por qué el protocolo C2 es difícil y por qué C5 tiene AUC aceptable pero log-loss alto;
5. generar tablas, figuras y listas de casos para decidir el siguiente experimento.

La calibración por protocolo individual se considera solo descriptiva: no sería aplicable cuando aparece un protocolo nuevo sin etiquetas.

## 0. Dependencias

In [ ]:
# Descomente únicamente si falta alguna dependencia.
# %pip install numpy pandas scipy scikit-learn matplotlib seaborn

## 1. Librerías y configuración

In [ ]:
from pathlib import Path
import os
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid')
SEED = 20260910
EPS = 1e-6
np.random.seed(SEED)

In [ ]:
DATA_ROOT = Path(r'C:\Users\DELL\OneDrive\Escritorio\kaggle\parkinson')
PROJECT_DIR = DATA_ROOT / 'latent_protocol_cv'
CNN_DIR = PROJECT_DIR / 'cnn3d_local_2ch_v1'
OOF_CSV = CNN_DIR / 'protocol_grouped_oof.csv'
FOLDS_CSV = PROJECT_DIR / 'outputs' / 'train_protocol_folds.csv'
ROI_CSV = PROJECT_DIR / 'roi_adaptive_v3' / 'roi_features.csv'
PREPROCESS_DIR = PROJECT_DIR / 'preprocessed_96x96x64_v2'
OUTPUT_DIR = PROJECT_DIR / 'diagnostico_calibracion_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in (OOF_CSV, FOLDS_CSV, ROI_CSV, PREPROCESS_DIR):
    print(path, '| existe:', path.exists())
    if not path.exists():
        raise FileNotFoundError(path)
print('Salidas:', OUTPUT_DIR)

## 2. Integridad y unión de resultados

In [ ]:
oof = pd.read_csv(OOF_CSV)
folds = pd.read_csv(FOLDS_CSV)
roi = pd.read_csv(ROI_CSV)

required_oof = {'uid', 'target', 'protocol_cluster', 'fold', 'prediction'}
missing_columns = required_oof - set(oof.columns)
if missing_columns:
    raise KeyError(f'Faltan columnas OOF: {sorted(missing_columns)}')
if oof['uid'].duplicated().any():
    raise ValueError('Hay UID duplicados en las predicciones OOF.')
if len(oof) != 1362:
    raise ValueError(f'Se esperaban 1,362 predicciones y se encontraron {len(oof):,}.')
if oof['prediction'].isna().any() or not oof['prediction'].between(0, 1).all():
    raise ValueError('Existen predicciones faltantes o fuera de [0,1].')
if set(oof['target'].unique()) - {0, 1}:
    raise ValueError('La variable target no es binaria 0/1.')

check_cols = ['uid', 'target', 'protocol_cluster', 'fold']
check = oof[check_cols].merge(folds[check_cols], on='uid', suffixes=('_oof', '_folds'), validate='one_to_one')
for col in check_cols[1:]:
    if not (check[f'{col}_oof'].to_numpy() == check[f'{col}_folds'].to_numpy()).all():
        raise ValueError(f'Inconsistencia entre OOF y manifiesto: {col}.')

roi_keep = [c for c in roi.columns if c == 'uid' or c not in oof.columns]
data = oof.merge(roi[roi_keep], on='uid', how='left', validate='one_to_one')
data['prediction'] = data['prediction'].clip(EPS, 1-EPS)
data['logit_raw'] = logit(data['prediction'])
data['case_logloss_raw'] = -(data['target']*np.log(data['prediction']) + (1-data['target'])*np.log(1-data['prediction']))
data['absolute_error_raw'] = np.abs(data['target']-data['prediction'])
data['predicted_class_raw'] = (data['prediction'] >= .5).astype(int)
data['correct_raw'] = data['predicted_class_raw'].eq(data['target'])
data['processed_path'] = data['uid'].map(lambda u: str((PREPROCESS_DIR/f'{u}.npz').resolve()))

print('N:', len(data), '| positivos:', int(data.target.sum()), '| prevalencia:', round(data.target.mean(), 4))
print('Folds:', sorted(data.fold.unique()), '| clusters:', sorted(data.protocol_cluster.unique()))
display(data.head())

## 3. Funciones de evaluación de probabilidades

In [ ]:
def safe_auc(y, p):
    return roc_auc_score(y, p) if np.unique(y).size == 2 else np.nan

def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y); p = np.asarray(p)
    bins = np.linspace(0, 1, n_bins+1)
    ids = np.clip(np.digitize(p, bins[1:-1], right=True), 0, n_bins-1)
    ece = 0.0
    for b in range(n_bins):
        mask = ids == b
        if mask.any():
            ece += mask.mean()*abs(y[mask].mean()-p[mask].mean())
    return float(ece)

def probability_metrics(frame, prediction_col, group_cols=None):
    group_cols = group_cols or []
    grouped = frame.groupby(group_cols, dropna=False) if group_cols else [((), frame)]
    rows = []
    for key, part in grouped:
        key = key if isinstance(key, tuple) else (key,)
        y = part.target.astype(int).to_numpy()
        p = part[prediction_col].clip(EPS, 1-EPS).to_numpy()
        row = dict(zip(group_cols, key))
        row.update(n=len(part), prevalence=y.mean(), mean_prediction=p.mean(),
                   logloss=log_loss(y,p), auc=safe_auc(y,p),
                   brier=brier_score_loss(y,p), ece_10=expected_calibration_error(y,p,10))
        rows.append(row)
    return pd.DataFrame(rows)

raw_global = probability_metrics(data, 'prediction')
raw_by_cluster = probability_metrics(data, 'prediction', ['protocol_cluster'])
raw_by_fold = probability_metrics(data, 'prediction', ['fold'])
display(raw_global)
display(raw_by_cluster)
raw_by_cluster.to_csv(OUTPUT_DIR/'metricas_crudas_por_cluster.csv', index=False)
raw_by_fold.to_csv(OUTPUT_DIR/'metricas_crudas_por_fold.csv', index=False)

## 4. Curvas de calibración crudas

In [ ]:
def reliability_table(frame, prediction_col, n_bins=10):
    p = frame[prediction_col].clip(EPS,1-EPS)
    # qcut evita muchos bins vacíos cuando las probabilidades están concentradas.
    bins = pd.qcut(p, q=min(n_bins, p.nunique()), duplicates='drop')
    out = frame.assign(_bin=bins).groupby('_bin', observed=True).agg(
        n=('target','size'), observed=('target','mean'), predicted=(prediction_col,'mean'))
    return out.reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for ax, (cluster, part) in zip(axes.flat, data.groupby('protocol_cluster')):
    rel = reliability_table(part, 'prediction', 8)
    ax.plot(rel.predicted, rel.observed, marker='o')
    ax.plot([0,1],[0,1],'--',color='gray')
    ax.set_title(f'C{cluster} | n={len(part)}')
    ax.set_xlabel('Probabilidad media'); ax.set_ylabel('Frecuencia observada')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'calibracion_cruda_por_cluster.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Calibración cruzada sin fuga

Se comparan dos calibradores globales:

- **Temperatura:** transforma el logit como $z/T$; conserva el orden y el AUC.
- **Platt:** estima intercepto y pendiente $a+bz$; puede corregir además un desplazamiento global.

Para cada fold externo, el calibrador se ajusta con las predicciones OOF de los otros cuatro folds y se aplica al fold excluido. Por tanto, cada paciente es calibrado sin utilizar su etiqueta ni las etiquetas de su fold.

In [ ]:
def fit_temperature(y, logits):
    y = np.asarray(y, dtype=int); logits = np.asarray(logits, dtype=float)
    objective = lambda log_t: log_loss(y, expit(logits/np.exp(log_t)), labels=[0,1])
    result = minimize_scalar(objective, bounds=(-3,3), method='bounded')
    return float(np.exp(result.x))

data['prediction_temperature_cf'] = np.nan
data['prediction_platt_cf'] = np.nan
calibration_rows = []

for fold in sorted(data.fold.unique()):
    train_mask = data.fold.ne(fold)
    valid_mask = data.fold.eq(fold)
    y_train = data.loc[train_mask, 'target'].astype(int).to_numpy()
    z_train = data.loc[train_mask, 'logit_raw'].to_numpy()
    z_valid = data.loc[valid_mask, 'logit_raw'].to_numpy()

    temperature = fit_temperature(y_train, z_train)
    data.loc[valid_mask, 'prediction_temperature_cf'] = expit(z_valid/temperature)

    platt = LogisticRegression(C=1e6, solver='lbfgs', max_iter=5000, random_state=SEED)
    platt.fit(z_train.reshape(-1,1), y_train)
    data.loc[valid_mask, 'prediction_platt_cf'] = platt.predict_proba(z_valid.reshape(-1,1))[:,1]
    calibration_rows.append({'fold_aplicado':fold, 'n_fit':int(train_mask.sum()),
                             'n_aplicado':int(valid_mask.sum()), 'temperature':temperature,
                             'platt_intercept':float(platt.intercept_[0]),
                             'platt_slope':float(platt.coef_[0,0])})

if data[['prediction_temperature_cf','prediction_platt_cf']].isna().any().any():
    raise RuntimeError('La calibración cruzada quedó incompleta.')

calibration_parameters = pd.DataFrame(calibration_rows)
display(calibration_parameters)
calibration_parameters.to_csv(OUTPUT_DIR/'parametros_calibracion_cruzada.csv', index=False)

In [ ]:
comparison_rows = []
for method, col in [('Cruda','prediction'), ('Temperatura cruzada','prediction_temperature_cf'),
                    ('Platt cruzado','prediction_platt_cf')]:
    row = probability_metrics(data, col).iloc[0].to_dict()
    row['metodo'] = method
    comparison_rows.append(row)
calibration_comparison = pd.DataFrame(comparison_rows)[['metodo','n','logloss','auc','brier','ece_10','mean_prediction','prevalence']]
display(calibration_comparison.style.format({c:'{:.6f}' for c in calibration_comparison.columns if c not in ['metodo','n']}))
calibration_comparison.to_csv(OUTPUT_DIR/'comparacion_calibracion_global.csv', index=False)

best_method = calibration_comparison.sort_values(['logloss','brier']).iloc[0]['metodo']
print('Mejor método por log-loss OOF cruzado:', best_method)

In [ ]:
by_cluster_all = []
for method, col in [('Cruda','prediction'), ('Temperatura cruzada','prediction_temperature_cf'),
                    ('Platt cruzado','prediction_platt_cf')]:
    tmp = probability_metrics(data, col, ['protocol_cluster'])
    tmp.insert(1, 'metodo', method)
    by_cluster_all.append(tmp)
calibration_by_cluster = pd.concat(by_cluster_all, ignore_index=True)
display(calibration_by_cluster)
calibration_by_cluster.to_csv(OUTPUT_DIR/'comparacion_calibracion_por_cluster.csv', index=False)

fig, axes = plt.subplots(1,2,figsize=(13,4.5))
sns.barplot(data=calibration_by_cluster, x='protocol_cluster', y='logloss', hue='metodo', ax=axes[0])
sns.barplot(data=calibration_by_cluster, x='protocol_cluster', y='brier', hue='metodo', ax=axes[1])
axes[0].set_title('Log-loss por cluster'); axes[1].set_title('Brier por cluster')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'comparacion_calibracion_por_cluster.png',dpi=160,bbox_inches='tight')
plt.show()

## 6. Distribución de predicciones: atención especial a C2 y C5

In [ ]:
fig, axes = plt.subplots(2,3,figsize=(14,8),sharex=True)
for ax,(cluster,part) in zip(axes.flat,data.groupby('protocol_cluster')):
    sns.histplot(data=part,x='prediction',hue='target',bins=np.linspace(0,1,21),
                 stat='density',common_norm=False,element='step',fill=False,ax=ax,palette=['#2878B5','#D9534F'])
    ax.axvline(part.target.mean(),color='black',linestyle=':',linewidth=1)
    ax.set_title(f'C{cluster} | AUC={safe_auc(part.target,part.prediction):.3f}')
    ax.set_xlabel('Probabilidad patológica')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'distribucion_predicciones_por_cluster.png',dpi=160,bbox_inches='tight')
plt.show()

In [ ]:
focus = data[data.protocol_cluster.isin([2,5])].copy()
focus_summary = focus.groupby(['protocol_cluster','target']).agg(
    n=('uid','size'), prediction_mean=('prediction','mean'), prediction_median=('prediction','median'),
    prediction_std=('prediction','std'), case_logloss_mean=('case_logloss_raw','mean'),
    error_absolute_mean=('absolute_error_raw','mean'), accuracy=('correct_raw','mean')).reset_index()
display(focus_summary)
focus_summary.to_csv(OUTPUT_DIR/'resumen_C2_C5_por_etiqueta.csv',index=False)

worst_cases = (focus.sort_values(['protocol_cluster','case_logloss_raw'],ascending=[True,False])
               .groupby('protocol_cluster',group_keys=False).head(20))
show_cols = [c for c in ['uid','protocol_cluster','fold','target','prediction','case_logloss_raw','z_peak',
                          'midline_x','pair_half_distance','left_contrast','right_contrast','bilateral_score',
                          'score_prominence','boundary_distance','y_boundary_distance'] if c in worst_cases.columns]
display(worst_cases[show_cols])
worst_cases.to_csv(OUTPUT_DIR/'peores_20_casos_C2_C5.csv',index=False)

## 7. Relación entre errores y características de la ROI

In [ ]:
candidate_features = [c for c in ['z_peak','midline_x','pair_half_distance','peak_distance',
    'left_contrast','right_contrast','bilateral_score','score_prominence','boundary_distance',
    'y_boundary_distance','left_x','left_y','right_x','right_y'] if c in data.columns]

correlation_rows=[]
for cluster in [2,5]:
    part=data[data.protocol_cluster.eq(cluster)]
    for feature in candidate_features:
        valid=part[[feature,'case_logloss_raw','absolute_error_raw']].dropna()
        if len(valid)>3 and valid[feature].nunique()>1:
            correlation_rows.append({'protocol_cluster':cluster,'feature':feature,
                'spearman_case_logloss':valid[feature].corr(valid.case_logloss_raw,method='spearman'),
                'spearman_absolute_error':valid[feature].corr(valid.absolute_error_raw,method='spearman')})
error_correlations=pd.DataFrame(correlation_rows)
display(error_correlations.reindex(error_correlations.spearman_case_logloss.abs().sort_values(ascending=False).index).head(20))
error_correlations.to_csv(OUTPUT_DIR/'correlaciones_error_ROI_C2_C5.csv',index=False)

if len(error_correlations):
    pivot=error_correlations.pivot(index='feature',columns='protocol_cluster',values='spearman_case_logloss')
    plt.figure(figsize=(6, max(4,.4*len(pivot))))
    sns.heatmap(pivot,annot=True,fmt='.2f',cmap='coolwarm',center=0,vmin=-1,vmax=1)
    plt.title('Correlación Spearman con pérdida individual')
    plt.tight_layout();plt.savefig(OUTPUT_DIR/'correlacion_error_ROI_C2_C5.png',dpi=160,bbox_inches='tight');plt.show()

## 8. Auditoría visual automática de los peores casos

In [ ]:
def crop2d(image, cx, cy, half=24):
    out=np.zeros((2*half,2*half),dtype=np.float32)
    x0=int(round(cx))-half; y0=int(round(cy))-half
    xs=slice(max(0,x0),min(image.shape[0],x0+2*half)); ys=slice(max(0,y0),min(image.shape[1],y0+2*half))
    dx=max(0,-x0); dy=max(0,-y0)
    out[dx:dx+(xs.stop-xs.start),dy:dy+(ys.stop-ys.start)]=image[xs,ys]
    return out

def plot_worst_cluster_cases(frame, cluster, n=8):
    selected=frame[frame.protocol_cluster.eq(cluster)].nlargest(n,'case_logloss_raw')
    fig,axes=plt.subplots(2,n,figsize=(2.25*n,4.8))
    for j,(_,row) in enumerate(selected.iterrows()):
        path=Path(row.processed_path)
        if not path.exists():
            axes[0,j].text(.5,.5,'Archivo no encontrado',ha='center'); axes[0,j].axis('off');axes[1,j].axis('off');continue
        with np.load(path) as saved: volume=saved['volume'].astype(np.float32)
        z=int(np.clip(round(row.z_peak),0,volume.shape[2]-1))
        local=crop2d(volume[:,:,z],row.midline_x,(row.left_y+row.right_y)/2,half=24)
        asym=np.abs(local-local[::-1,:])
        axes[0,j].imshow(local.T,cmap='magma',origin='lower');axes[1,j].imshow(asym.T,cmap='viridis',origin='lower')
        axes[0,j].set_title(f'{row.uid} | y={int(row.target)} p={row.prediction:.2f}\nLL={row.case_logloss_raw:.2f}',fontsize=8)
        axes[1,j].set_title('|I − espejo|',fontsize=8)
        axes[0,j].axis('off');axes[1,j].axis('off')
    fig.suptitle(f'C{cluster}: casos con mayor log-loss',y=1.02)
    plt.tight_layout();plt.savefig(OUTPUT_DIR/f'peores_casos_C{cluster}.png',dpi=160,bbox_inches='tight');plt.show()

plot_worst_cluster_cases(data,2,8)
plot_worst_cluster_cases(data,5,8)

## 9. Regla automática de decisión

In [ ]:
raw_row=calibration_comparison.query("metodo == 'Cruda'").iloc[0]
best_cal=calibration_comparison.sort_values('logloss').iloc[0]
c2=raw_by_cluster.query('protocol_cluster == 2').iloc[0]
c5=raw_by_cluster.query('protocol_cluster == 5').iloc[0]
c5_cal=(calibration_by_cluster.query('protocol_cluster == 5')
        .sort_values('logloss').iloc[0])

decision={
    'raw_logloss_oof':float(raw_row.logloss),
    'raw_auc_oof':float(raw_row.auc),
    'raw_brier_oof':float(raw_row.brier),
    'best_cross_fitted_calibration':str(best_cal.metodo),
    'best_calibrated_logloss_oof':float(best_cal.logloss),
    'calibration_logloss_gain':float(raw_row.logloss-best_cal.logloss),
    'cluster_2_auc':float(c2.auc),
    'cluster_2_logloss':float(c2.logloss),
    'cluster_5_auc':float(c5.auc),
    'cluster_5_raw_logloss':float(c5.logloss),
    'cluster_5_best_calibrated_logloss':float(c5_cal.logloss),
}

if decision['calibration_logloss_gain'] >= .01:
    calibration_conclusion='La calibración cruzada produce una mejora relevante y debe conservarse para el modelo final.'
else:
    calibration_conclusion='La calibración cruzada no produce una mejora relevante; no debe añadirse por ahora.'
if c2.auc < .65:
    c2_conclusion='C2 requiere un experimento estructural: modelo 2.5D sobre cortes axiales o representación específica del protocolo.'
else:
    c2_conclusion='C2 conserva discriminación suficiente; priorizar calibración y análisis de casos antes de cambiar la arquitectura.'
if c5.auc >= .70 and (c5.logloss-c5_cal.logloss) >= .01:
    c5_conclusion='C5 discrimina y mejora con calibración; no necesita todavía una arquitectura especializada.'
elif c5.auc >= .70:
    c5_conclusion='C5 discrimina, pero la calibración global cruzada no corrige suficientemente sus probabilidades.'
else:
    c5_conclusion='C5 también presenta un problema de discriminación y requiere revisión estructural.'

decision.update(calibration_conclusion=calibration_conclusion,c2_conclusion=c2_conclusion,c5_conclusion=c5_conclusion)
print(calibration_conclusion);print(c2_conclusion);print(c5_conclusion)
with open(OUTPUT_DIR/'decision_diagnostica.json','w',encoding='utf-8') as f:
    json.dump(decision,f,ensure_ascii=False,indent=2)
display(pd.Series(decision,name='resultado').to_frame())

## 10. Exportación final

In [ ]:
export_cols=['uid','target','protocol_cluster','fold','prediction','prediction_temperature_cf',
             'prediction_platt_cf','case_logloss_raw','absolute_error_raw','correct_raw']
data[export_cols].to_csv(OUTPUT_DIR/'predicciones_oof_calibradas_y_diagnostico.csv',index=False)

readme=f'''Diagnóstico CNN 3D local de dos canales
N={len(data)}
Log-loss OOF crudo={raw_row.logloss:.6f}
AUC OOF crudo={raw_row.auc:.6f}
Brier OOF crudo={raw_row.brier:.6f}
Mejor calibración cruzada={best_cal.metodo}
Log-loss OOF calibrado={best_cal.logloss:.6f}
Mejora de log-loss={raw_row.logloss-best_cal.logloss:.6f}

{calibration_conclusion}
{c2_conclusion}
{c5_conclusion}
'''
(OUTPUT_DIR/'RESUMEN.txt').write_text(readme,encoding='utf-8')
print(readme)
print('Archivos guardados en:',OUTPUT_DIR)

## Cómo interpretar el resultado

- La calibración se acepta solo si reduce el log-loss OOF cruzado, idealmente también Brier y ECE, sin modificar el AUC.
- Una mejora aparente obtenida calibrando y evaluando sobre los mismos 1,362 pacientes no se considera evidencia válida.
- AUC bajo en C2 indica un problema de representación/discriminación, no solo de calibración.
- AUC aceptable con log-loss alto en C5 indica que existe señal, pero las probabilidades pueden estar desplazadas o tener confianza incorrecta.
- No se debe escoger una calibración diferente para cada cluster utilizando sus propias etiquetas si el objetivo es simular protocolos nuevos.